In [1]:
import jno
import jax
import optax
import equinox as eqx
import numpy as np

In [2]:
jno.setup('./')

'/home/users/armbrust/projects/jno/runs/jno'

In [3]:
dom = jno.domain.rect(mesh_size=0.025, compute_mesh_connectivity=False)
x, y, _ = dom.variable('interior')

2026-03-24 13:16:50,923 - INFO - Loaded mesh from <function Geometries.rect.<locals>.construct at 0x7f01dae87880>: 1941 points
2026-03-24 13:16:51,000 - INFO - Sampled 1935 points for 'interior'


In [4]:
net = jno.nn.mlp(2, key=jax.random.PRNGKey(0))

In [5]:
all_false = jax.tree_util.tree_map(lambda _: False, net.module)
param_mask = eqx.tree_at(lambda m: (m.hidden_layers[0].weight, m.hidden_layers[0].bias), all_false, (True, True))

In [ ]:
net = jno.nn.mlp(2, key=jax.random.PRNGKey(0))
net.optimizer(optax.adam(optax.schedules.cosine_decay_schedule(1e-3, 1000)))
net.mask(param_mask).lora(rank=4)
net.optimizer(optax.adamw(optax.schedules.cosine_decay_schedule(1e-3, 1000)))

Model(MLP)

In [7]:
print(f'Frozen: {np.mean(net.module.hidden_layers[1].weight):.12f}')
print(f'Not Frozen: {np.mean(net.module.hidden_layers[0].weight):.12f}')

Frozen: 0.000530750724
Not Frozen: -0.007068308070


In [8]:
u = net(x, y) * (1 - x) * (1 - y)
pde = jno.np.laplacian(u, [x, y]) + 1

In [9]:
crux = jno.core(constraints=[pde.mse], domain=dom)

2026-03-24 13:16:52,246 - INFO - RNG seed: 21
2026-03-24 13:16:52,246 - INFO - Initializing Model/s and compiling constraints
2026-03-24 13:16:52,247 - INFO - Device mesh: Mesh('batch': 1, 'model': 1, axis_types=(Auto, Auto)) (shape: (1, 1))
2026-03-24 13:16:52,249 - INFO -   MLP: 4,417 parameters
2026-03-24 13:16:52,250 - INFO - Using 1 device(s): [CpuDevice(id=0)]


In [10]:
crux.solve(epochs=1000).plot('/home/users/armbrust/projects/runs/jno/th.png')

2026-03-24 13:16:52,253 - INFO - Paramax auto-unwrap enabled: wrappers are unwrapped before each forward evaluation
2026-03-24 13:16:52,730 - INFO - LoRA applied to model 2 (rank=4, alpha=1.0): 3 LoRALinear layers, Params: 4,417→5,126
2026-03-24 13:16:52,731 - INFO - Parameter summary:
2026-03-24 13:16:52,731 - INFO -     Trainable parameters:         5,126
2026-03-24 13:16:52,732 - INFO -     Frozen parameters:                0
2026-03-24 13:16:52,732 - INFO -     Total parameters:             5,126
2026-03-24 13:16:52,733 - INFO -     LoRA parameters:                709 (included in trainable)
2026-03-24 13:16:52,734 - INFO -     LoRA % of total:             13.83%
2026-03-24 13:16:53,151 - INFO - Constraint 0: Shape = (1, 1935, 1) → .mse() → (1,)
2026-03-24 13:16:53,168 - INFO - JIT compiling step function with mesh sharding — this might take a while
2026-03-24 13:16:54,124 - INFO - Epoch      0/1000| L:1.1855e+00 | C0: 1.1855e+00
2026-03-24 13:16:54,781 - INFO - Epoch    100/1000| 

In [11]:
print(f'Frozen: {np.mean(net.module.hidden_layers[1].weight):.12f}')
print(f'Not Frozen: {np.mean(net.module.hidden_layers[0].weight):.12f}')

Frozen: 0.000988631509
Not Frozen: -0.011892686598


In [12]:
net.unfreeze()
net.optimizer(optax.lbfgs(optax.schedules.cosine_decay_schedule(1e-3, 200)))
crux.solve(epochs=200).plot('/home/users/armbrust/projects/runs/jno/th2.png')

2026-03-24 13:17:06,565 - INFO - Paramax auto-unwrap enabled: wrappers are unwrapped before each forward evaluation
2026-03-24 13:17:07,469 - INFO - LoRA applied to model 2 (rank=4, alpha=1.0): 3 LoRALinear layers, Params: 4,417→5,126
2026-03-24 13:17:07,470 - INFO - Parameter summary:
2026-03-24 13:17:07,503 - INFO -     Trainable parameters:           709
2026-03-24 13:17:07,540 - INFO -     Frozen parameters:            4,417
2026-03-24 13:17:07,576 - INFO -     Total parameters:             5,126
2026-03-24 13:17:07,612 - INFO -     LoRA parameters:                709 (included in trainable)
2026-03-24 13:17:07,818 - INFO -     LoRA % of total:             13.83%
2026-03-24 13:17:08,425 - INFO - Constraint 0: Shape = (1, 1935, 1) → .mse() → (1,)
2026-03-24 13:17:08,427 - INFO - JIT compiling step function with mesh sharding — this might take a while
2026-03-24 13:17:12,183 - INFO - Epoch      0/200| L:5.5575e-03 | C0: 5.5575e-03
2026-03-24 13:17:13,303 - INFO - Epoch     20/200| L:

In [13]:
print(f'Frozen: {np.mean(net.module.hidden_layers[1].weight):.12f}')
print(f'Not Frozen: {np.mean(net.module.hidden_layers[0].weight):.12f}')

Frozen: 0.001026496291
Not Frozen: -0.009820567444
